# Loss of a single MOSFET

Loss and junction temperature of **one single MOSFET**, from the manufacturer data
of the part and of its driver.

**How to use it — 3 steps:**

1. Run the configuration cell (`Kernel → Restart & Run All` does the lot at once).
2. Pick the MOSFET, the driver and the operating point in the panel.
3. Click **Compute**.

> Reverse recovery is a field you fill in: the $Q_{rr}$ of the diode the turn-on
> forces to recover — the facing one, not this MOSFET's own. Leave it at 0 when
> there is none.

In [1]:
# --- Configuration: run this first -------------------------------------------
import base64
import sys
from pathlib import Path

# Walk up to the project root (the folder holding DATABASE/)
ROOT = Path.cwd()
while not (ROOT / "DATABASE").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import ipywidgets as W
import matplotlib.pyplot as plt
from IPython.display import HTML, clear_output, display

from DATABASE.db_driver_mosfet import DRIVER_LIBRARY, load_driver
from DATABASE.db_mosfet import MOSFET_LIBRARY, load_mosfet
from SRC.MOSFET.mosfet_loss import (
    OPERATING_POINT,
    loss_single_mosfet_at_temp,
    loss_thermal_iteration,
)
from SRC.MOSFET.mosfet_plot import (
    loss_table,
    plot_loss_breakdown,
    plot_thermal_iteration,
)
from SRC.OTHERS.plot import save_figure
from SRC.OTHERS.terminal import alert, blank, dataframe, kv, section, table, use_theme
from SRC.THERMAL.dissipator import *

#%matplotlib inline

# Figures rendered as SVG rather than as a screen-resolution bitmap: they stay
# sharp at any zoom, and copy out of the notebook as vector. Swap for "retina"
# (a 2x PNG) if a viewer chokes on the SVG or if you want a smaller .ipynb.
%config InlineBackend.figure_formats = ["svg"]

# Report palette, shared with the .py twin of this notebook.
#   "dark"  -> for a dark JupyterLab theme
#   "light" -> for a light one
#   "red" / "blue" / "vivid" -> accent themes, dark background only
#   "mono"  -> no colour at all
# Same five names as PLOT_THEME below, so a report and its figures match.
# Add your own in SRC/OTHERS/terminal.py: a theme is just a dict of styles.
THEME = "dark"
# matplotlib figures, independent of the report palette above.
#   "light" / "dark" -> the neutral pair
#   "red" / "blue" / "vivid" -> accent themes on a tinted black
PLOT_THEME = "light"

# --- Figure export, same settings as the .py twin ----------------------------
# Every Compute also writes the figures to OUTPUT/. "svg" / "pdf" are vector
# (sharp at any print size — that is what goes into a report or into LaTeX),
# "png" is only for what refuses anything else.
SAVE_PLOTS = False
PLOT_FORMATS = ("png", "svg", "pdf")
PLOT_DPI = 300  # raster resolution; ignored by svg / pdf
OUTPUT_DIR = ROOT / "OUTPUT"

use_theme(THEME)

kv("Project root", str(ROOT))
kv("MOSFETs available", ", ".join(sorted(MOSFET_LIBRARY)))
kv("Drivers available", ", ".join(sorted(DRIVER_LIBRARY)))

Project root                          c:\Users\tro\OneDrive - Danish Aerospace\Documents\Projects\Test\MyCalculator_V3

MOSFETs available                     BSC016N06NS

Drivers available                     UCC27714

---
## 1. Reference figure — the gate charge

Every switching time of the model comes out of this diagram. The charges
$Q_{g(th)}$, $Q_{gs}$, $Q_{gd}$ and the Miller plateau are what sets the switching
speed, hence the loss.

Pick the image to display from the list (anything sitting in `DOCUMENTS/`).

In [ ]:
# --- Reference image ---------------------------------------------------------
_MIME = {".png": "image/png", ".jpg": "image/jpeg", ".jpeg": "image/jpeg",
         ".webp": "image/webp", ".gif": "image/gif", ".svg": "image/svg+xml"}

_images = sorted(
    (p for p in (ROOT / "DOCUMENTS").rglob("*") if p.suffix.lower() in _MIME),
    key=lambda p: p.name,
)


def show_image(path, max_width=720):
    """Display an image by embedding it as base64 (works for .webp too)."""
    path = Path(path)
    if not path.exists():
        return HTML(f"<p style='color:#d03b3b'>Image not found: {path}</p>")
    mime = _MIME.get(path.suffix.lower(), "image/png")
    b64 = base64.b64encode(path.read_bytes()).decode()
    return HTML(
        f"<img src='data:{mime};base64,{b64}' "
        f"style='max-width:{max_width}px;width:100%;border-radius:6px'>"
    )


# Default: the gate charge diagram
_default = next((p for p in _images if "gate_charge" in p.name), _images[0] if _images else None)

image_dd = W.Dropdown(
    options=[(p.relative_to(ROOT).as_posix(), p) for p in _images],
    value=_default,
    description="Image:",
    style={"description_width": "80px"},
    layout=W.Layout(width="620px"),
)
image_out = W.Output()


def _refresh_image(_=None):
    with image_out:
        clear_output(wait=True)
        display(show_image(image_dd.value))


image_dd.observe(_refresh_image, names="value")
_refresh_image()
display(W.VBox([image_dd, image_out]))

---
## 2. The formulas

### 2.1 Gate loop — where the times come from

$$R_{G,tot} = R_{drv} + R_{ext} + R_{g,int}
\qquad
I_G = \frac{V_{drive} - V_{gate}}{R_{G,tot}}
\quad\text{(clamped to the driver peak current)}$$

Turn-on splits into two sub-intervals, and turn-off is its mirror image:

| Sub-interval | Charge moved | Gate voltage | What moves |
|---|---|---|---|
| $t_{ri}$ | $Q_{gs} - Q_{g(th)}$ | $\tfrac12(V_{th}+V_{pl})$ | the **current** rises, $V_{ds}$ stays high |
| $t_{fv}$ | $Q_{gd}$ | $V_{pl}$ (plateau) | the **voltage** falls |
| $t_{rv}$ | $Q_{gd}$ | $V_{pl}$ | the voltage rises again |
| $t_{fi}$ | $Q_{gs} - Q_{g(th)}$ | $\tfrac12(V_{th}+V_{pl})$ | the current falls |

$$t = \frac{Q_{\text{region}}}{I_G}
\qquad
t_{on} = t_{ri} + t_{fv}
\qquad
t_{off} = t_{rv} + t_{fi}$$

### 2.2 The six losses

$$\boxed{P_{cond} = R_{DS(on)}(T_j)\; I_{rms}^2}
\qquad
R_{DS(on)}(T_j) = R_{25}\left[1 + \alpha_R (T_j - 25)\right]$$

$$\boxed{P_{sw} = \tfrac12\left(V_{on} I_{on} t_{on} + V_{off} I_{off} t_{off}\right) f_{sw}}$$

$$\boxed{P_{oss} = \tfrac12\, C_{oss,er}(V_{on})\, V_{on}^2\, f_{sw}}
\qquad
C_{oss,er}(V) = \frac{2}{V^2}\int_0^{V} C_{oss}(v)\, v\, dv$$

$$\boxed{P_{body} = \underbrace{V_F I_{body} D_{body}}_{\text{dead-time conduction}}
+ \underbrace{Q_{rr}\, V_{on}\, f_{sw}}_{\text{reverse recovery}}}$$

**The $Q_{rr}$ is entered by hand, and it is the facing diode's.** The diode the
turn-on forces to recover is not this MOSFET's: it belongs to the opposite device
(parallel Schottky, discrete freewheel diode, another transistor). 

$$\boxed{P_{gate} = Q_g\, \Delta V_{gs}\, f_{sw}}
\qquad\text{split}\ \propto R:\quad
P_{g,int} = P_{gate}\frac{R_{g,int}}{R_{G,tot}}$$

**Only the internal share heats the die.** The driver and the external resistor
dissipate theirs outside the package:

$$\boxed{P_{total} = P_{cond} + P_{sw} + P_{oss} + P_{body} + P_{g,int}}$$

### 2.3 Switching slew rates

$$\left.\frac{di}{dt}\right|_{on} = \frac{I_{on}}{t_{ri}}
\qquad
\left.\frac{dv}{dt}\right|_{on} = \frac{V_{on}}{t_{fv}}
\qquad
\left.\frac{dv}{dt}\right|_{off} = \frac{V_{off}}{t_{rv}}$$

### 2.4 Thermal coupling

$R_{DS(on)}$ rises with $T_j$, and $T_j$ rises with the loss: neither can be computed
on its own. Fixed point, iterated to convergence:

$$\boxed{T_j^{(k+1)} = T_{amb} + R_{th}\; P_{total}\!\left(T_j^{(k)}\right)}$$

If it diverges, the design is in **thermal runaway** — the notebook says so.

### 2.5 Switched voltages — where the model is flexible

There is **no** "hard/soft switching" switch: you enter the voltage each edge actually
sweeps.

| Case | $V_{turn\,on}$ | $V_{turn\,off}$ |
|---|---|---|
| Hard switching | $V_{bus}$ | $V_{bus}$ |
| ZVS at turn-on | $0$ | $V_{bus}$ |
| ZVS on both edges | $0$ | $0$ |
| Snubbed at turn-off | $V_{bus}$ | a fraction of $V_{bus}$ |

Setting $V_{turn\,on} = 0$ kills $P_{oss}$ **and** the recovery term on its own.

---
## 3. Calculation panel

### 3.1 Cooling — one path per package face

Up to here $R_{th}$ was a number typed in by hand. It is now computed from the real
geometry: every package face that carries heat away opens its own **branch**, and the
branches add up in parallel.

```
                            ( T_j )
                               │
              ┌────────────────┴────────────────┐
        R_thJC(bottom)                    R_thJC(top)      ← datasheet, THERMAL_INFO.r_thjc
              │                                 │
        R_ext(bottom)                     R_ext(top)       ← Dissipator.get_rth()
              │                                 │
              └────────────────┬────────────────┘
                               │
                          ( T_ambient )
```

$$\boxed{R_{thJA} = \left[\;\sum_{\text{active faces}} \frac{1}{R_{thJC,f} + R_{ext,f}}\right]^{-1}}$$

That is exactly what `THERMAL_INFO.r_thja_value(external_paths)` does: it pairs each
$R_{ext}$ with its $R_{thJC}$ **by face name** (`"bottom"`, `"top"`), puts the two in
series, then all the results in parallel.

**Two families of $R_{ext}$**, both in `SRC/THERMAL/dissipator.py`:

| Class | What you enter | What for |
|---|---|---|
| `StandardDissipator` | manufacturer $R_{th}$ + $R_{TIM}$ | alu heatsink, thermal pad, baseplate |
| `PCBDissipator` | copper / substrate / via geometry | dissipation through the PCB itself |

The `PCBDissipator` puts two paths in parallel: direct convection off the plane on the
pad side, and crossing the substrate (bare FR4 ∥ thermal vias) to the opposite plane
and then convection. Its useful area is capped by the spreading radius
$r_{max} = 1.5\,\text{cm}\times\sqrt{t_{cu}/35\,\mu m}$: beyond that, extra copper buys
nothing, the heat never gets there.



In [3]:
_STD = {"description_width": "215px"}
_LYD = W.Layout(width="410px")


def _fd(desc, value, step=None, lo=0.0, hi=1e9):
    return W.BoundedFloatText(value=value, description=desc, style=_STD, layout=_LYD,
                              step=step, min=lo, max=hi)


# Common substrates: (k [W/mK], typical thickness [mm]). "Custom" touches nothing.
SUBSTRATES = {
    "Standard FR4": (0.3, 1.6),
    "IMS (alu base)": (2.0, 0.1),
    "DBC Al2O3": (25.0, 0.38),
    "DBC AlN": (170.0, 0.63),
    "Custom": (None, None),
}


class FaceDissipation:
    """Settings of one face: no dissipator, a PCB copper plane, or a heatsink."""

    def __init__(self, face, *, mode="none", a_cu=10.0, a_cu_other=10.0, a_pad=0.3,
                 t_cu=70.0, n_vias=24, r_th_std=5.0, h_conv=25.0):
        self.face = face

        self.mode = W.Dropdown(
            options=[("None — uncooled face", "none"),
                     ("PCB — copper plane", "pcb"),
                     ("Heatsink — datasheet R_th", "std")],
            value=mode, description="Dissipation:", style=_STD, layout=_LYD,
        )

        # -- PCBDissipator
        self.substrate = W.Dropdown(options=list(SUBSTRATES), value="Standard FR4",
                                    description="Substrate:", style=_STD, layout=_LYD)
        self.a_cu = _fd("Total copper this side [cm²]:", a_cu, 0.5, lo=0.01)
        self.a_cu_other = _fd("Copper on far side [cm²] (0 = none):", a_cu_other, 0.5)
        self.a_pad = _fd("Thermal pad area [cm²]:", a_pad, 0.05, lo=0.01)
        self.t_cu = _fd("Copper thickness [µm] (35 = 1 oz):", t_cu, 35.0, lo=1.0)
        self.e_pcb = _fd("Substrate thickness [mm]:", 1.6, 0.1, lo=0.01)
        self.k_pcb = _fd("Substrate k [W/(m·K)]:", 0.3, 0.1, lo=0.01)
        self.k_cond = _fd("Conductor k [W/(m·K)]:", 385.0, 10.0, lo=1.0)
        self.n_vias = W.BoundedIntText(value=n_vias, description="Number of vias:",
                                       style=_STD, layout=_LYD, min=0, max=10000)
        self.d_via = _fd("Via diameter [mm]:", 0.3, 0.05, lo=0.01)
        self.plating = _fd("Via plating [µm]:", 25.0, 5.0, lo=0.1)
        self.filled = W.Checkbox(value=False, description="Filled vias (solid copper)",
                                 indent=False, layout=W.Layout(width="410px"))
        self.h_conv = _fd("h convection + radiation [W/(m²·K)]:", h_conv, 1.0, lo=0.1)

        # -- StandardDissipator
        self.r_th = _fd("Heatsink R_th [°C/W]:", r_th_std, 0.5, lo=0.001)
        self.r_tim = _fd("TIM interface R_th [°C/W]:", 0.5, 0.1)

        self._box_pcb = W.VBox([self.substrate, self.a_cu, self.a_cu_other, self.a_pad,
                                self.t_cu, self.e_pcb, self.k_pcb, self.k_cond,
                                self.n_vias, self.d_via, self.plating, self.filled,
                                self.h_conv])
        self._box_std = W.VBox([self.r_th, self.r_tim])
        self.box = W.VBox([self.mode, self._box_pcb, self._box_std],
                          layout=W.Layout(margin="8px 0 0 0"))

        self.mode.observe(self._sync_mode, names="value")
        self.substrate.observe(self._sync_substrate, names="value")
        self._sync_mode()

    # -- conditional display: only show the fields that are in use
    def _sync_mode(self, _=None):
        self._box_pcb.layout.display = "" if self.mode.value == "pcb" else "none"
        self._box_std.layout.display = "" if self.mode.value == "std" else "none"

    def _sync_substrate(self, _=None):
        k, e = SUBSTRATES[self.substrate.value]
        if k is not None:
            self.k_pcb.value, self.e_pcb.value = k, e

    def build(self):
        """Instantiate the matching Dissipator, or None if the face is inactive."""
        if self.mode.value == "none":
            return None
        placement = Placement(self.face)
        if self.mode.value == "std":
            return StandardDissipator(
                name=f"Heatsink {self.face}", placement=placement,
                R_th=self.r_th.value, R_tim=self.r_tim.value,
            )
        return PCBDissipator(
            name=f"Copper plane {self.face}", placement=placement,
            A_cu_total_side_cm2=self.a_cu.value,
            A_cu_other_side_cm2=(self.a_cu_other.value or None),
            A_pad_mosfet_cm2=self.a_pad.value,
            copper_thickness_um=self.t_cu.value,
            pcb_thickness_mm=self.e_pcb.value,
            pcb_k_W_mK=self.k_pcb.value,
            k_conductor_W_mK=self.k_cond.value,
            n_vias=self.n_vias.value,
            via_diameter_mm=self.d_via.value,
            via_plating_um=self.plating.value,
            via_filled=self.filled.value,
            h_conv_eff_W_m2K=self.h_conv.value,
        )


# One face per Placement value — hence per R_thJC declared in the datasheet.
# Default: 10 cm² copper plane on both sides, 24 vias, light airflow (h = 25).
FACES = {
    Placement.BOTTOM.value: FaceDissipation(Placement.BOTTOM.value, mode="pcb"),
    Placement.TOP.value: FaceDissipation(
        Placement.TOP.value, mode="none", a_cu=2.0, a_cu_other=0.0, n_vias=0,
        r_th_std=5.0,
    ),
}

faces_tab = W.Tab(children=[f.box for f in FACES.values()], layout=W.Layout(width="470px"))
for i, name in enumerate(FACES):
    faces_tab.set_title(i, f"{name.capitalize()} face")


def dissipators():
    """The active dissipators, in face order."""
    return [d for d in (f.build() for f in FACES.values()) if d is not None]


def external_paths():
    """The shape THERMAL_INFO.r_thja_value expects: [(R_ext, face name), ...]."""
    return [(d.get_rth(), d.placement.value) for d in dissipators()]

In [4]:
# --- Control panel -----------------------------------------------------------
_ST = {"description_width": "185px"}
_LY = W.Layout(width="380px")


def _f(desc, value, step=None, lo=0.0, hi=1e9):
    """Bounded numeric field. lo/hi keep nonsense values out."""
    return W.BoundedFloatText(value=value, description=desc, style=_ST, layout=_LY,
                              step=step, min=lo, max=hi)


# Components
w_mosfet = W.Dropdown(options=sorted(MOSFET_LIBRARY), description="MOSFET:",
                      style=_ST, layout=_LY)
w_driver = W.Dropdown(options=sorted(DRIVER_LIBRARY), description="Driver:",
                      style=_ST, layout=_LY)

# Operating point
w_von = _f("Voltage swept at turn-on [V]:", 48.0, 1.0)
w_voff = _f("Voltage swept at turn-off [V]:", 48.0, 1.0)
w_irms = _f("RMS current I_rms [A]:", 15.8, 0.5)
w_ion = _f("Current commutated at turn-on [A]:", 25.0, 0.5)
w_ioff = _f("Current commutated at turn-off [A]:", 25.0, 0.5)
w_fsw = _f("Switching frequency [kHz]:", 100.0, 10.0)

# Gate loop
w_rgon = _f("External gate R, turn-on [Ω]:", 2.2, 0.1)
w_rgoff = _f("External gate R, turn-off [Ω]:", 1.0, 0.1)

# Body diode: dead-time conduction, nothing more. Reverse recovery is entered
# separately — that diode is not the one that recovers.
w_ibody = _f("Diode current, dead time [A]:", 0.0, 0.5)
w_dbody = _f("Diode duty cycle [%]:", 0.0, 0.5, hi=100.0)

# Q_rr — the FACING diode's, entered by hand. No "auto" mode: a lone MOSFET
# knows nothing about the diode its turn-on forces to recover, so there is
# nothing to infer. 0 = no recovery (ZCS, ZVS, GaN).
w_qrr = _f("Q_rr of the facing diode [nC]:", 0.0, 1.0)

# Thermal
w_tj = _f("Evaluation Tj [°C]:", 100.0, 5.0, lo=-40.0, hi=300.0)
w_tamb = _f("Ambient temperature [°C]:", 40.0, 5.0, lo=-40.0, hi=200.0)
w_th_mode = W.Dropdown(
    options=[("Dissipators — computed per face", "dissip"),
             ("Datasheet R_thJA", "datasheet"),
             ("R_th entered by hand", "manual")],
    value="dissip", description="R_th source:", style=_ST, layout=_LY,
)
w_rth = _f("R_th junction→ambient [°C/W]:", 20.0, 1.0)
w_rth.disabled = True

_TH_LABEL = {"dissip": "dissipators", "datasheet": "datasheet", "manual": "entered"}


def _toggle_rth(_=None):
    w_rth.disabled = w_th_mode.value != "manual"
    faces_tab.layout.display = "" if w_th_mode.value == "dissip" else "none"


w_th_mode.observe(_toggle_rth, names="value")

w_go = W.Button(description="Compute", button_style="primary",
                icon="calculator", layout=W.Layout(width="200px", height="38px"))
out = W.Output()


def _group(title, widgets):
    return W.VBox([W.HTML(f"<b style='font-size:13px'>{title}</b>"), *widgets],
                  layout=W.Layout(margin="0 28px 14px 0"))


panel = W.VBox([
    W.HBox([
        _group("Components", [w_mosfet, w_driver]),
        _group("Operating point", [w_von, w_voff, w_fsw]),
    ]),
    W.HBox([
        _group("Currents", [w_irms, w_ion, w_ioff]),
        _group("Gate loop", [w_rgon, w_rgoff]),
    ]),
    W.HBox([
        _group("Body diode / recovery", [w_ibody, w_dbody, w_qrr]),
        _group("Thermal", [w_tj, w_tamb, w_th_mode, w_rth]),
        _group("Cooling, per face", [faces_tab]),
    ], layout=W.Layout(flex_flow="row wrap")),
    w_go,
    out,
])


# ============================================================================ #
#  Model inputs and sanity checks
# ============================================================================ #


# An alert is a (level, message) pair: "stop" aborts the calculation, "warn"
# only goes along with it. Same convention as the .py twin.
def stop(message):
    return "stop", message


def warn(message):
    return "warn", message


def _operating_point():
    return OPERATING_POINT(
        v_turn_on=w_von.value,
        v_turn_off=w_voff.value,
        i_rms=w_irms.value,
        f_sw=w_fsw.value * 1e3,
        i_on=w_ion.value,
        i_off=w_ioff.value,
        r_g_ext_on=w_rgon.value,
        r_g_ext_off=w_rgoff.value,
        i_body=w_ibody.value,
        d_body=w_dbody.value / 100.0,
        # Never None: the model's "this MOSFET's own body diode" fallback makes
        # no sense here, it is the facing diode that recovers.
        q_rr_opposite=w_qrr.value * 1e-9,
    )


def _fmt_r(value):
    """A thermal resistance, infinity included (missing via = open path)."""
    return "inf" if value > 1e6 else f"{value:.1f}"


def _thermal_resistance(mosfet):
    """
    (r_th, alerts, active dissipators) for the source selected in the panel.

    r_th = None lets loss_thermal_iteration fall back to the datasheet R_thJA.
    """
    if w_th_mode.value == "manual":
        return w_rth.value, [], []
    if w_th_mode.value == "datasheet":
        return None, [], []

    try:
        active = dissipators()
    except ValueError as err:  # geometry rejected by pydantic (ValidationError)
        detail = str(err).split("Value error, ")[-1].split(" [type=")[0].strip()
        return None, [stop(f"Invalid dissipator geometry — {detail}")], []

    if not active:
        return None, [warn(
            "No cooled face: falling back to the datasheet R_thJA "
            f"({mosfet.thermal.r_thja[0]:.0f} °C/W). Enable at least one face "
            'under "Cooling, per face".'
        )], []

    available = [face for _, face in mosfet.thermal.r_thjc]
    missing = [d.placement.value for d in active if d.placement.value not in available]
    if missing:
        return None, [stop(
            f'{w_mosfet.value} has no R_thJC for the "{missing[0]}" face '
            f"(available: {', '.join(available)}). Fill in `r_thjc` in "
            "MOSFET_LIBRARY."
        )], []

    alerts = []
    pcbs = [d for d in active if isinstance(d, PCBDissipator)]

    # A PCB only has two planes: declaring both sides counts the copper twice.
    if len(pcbs) > 1 and all(p.A_cu_other_side_cm2 for p in pcbs):
        alerts.append(warn(
            'Both faces are in "PCB" mode and each declares an opposite plane: '
            'the same copper is counted twice. Set "Copper on far side" to 0 on '
            "one of them."
        ))
    # Copper beyond the spreading radius: paid for in board area, useless thermally.
    for p in pcbs:
        if p.A_cu_effective_side_cm2 < p.A_cu_total_side_cm2 - 1e-9:
            alerts.append(warn(
                f"{p.name}: only {p.A_cu_effective_side_cm2:.1f} cm² of the "
                f"{p.A_cu_total_side_cm2:.1f} cm² declared take part (spreading "
                f"radius at {p.copper_thickness_um:.0f} µm). Widening the plane "
                "will not help any more: thicker copper, vias, or airflow."
            ))

    paths = [(d.get_rth(), d.placement.value) for d in active]
    return mosfet.thermal.r_thja_value(paths), alerts, active


def _checks(mosfet, op):
    """Consistency of the operating point against the part's ratings."""
    alerts = []
    v_max_coss = mosfet.c_oss.vds_points[-1]
    v_sw = max(op.v_turn_on, op.v_turn_off)

    if v_sw > mosfet.v_dss_max:
        alerts.append(stop(
            f"Switched voltage {v_sw:.0f} V above the V_DSS rating of the "
            f"{mosfet.component_info.part_number} ({mosfet.v_dss_max:.0f} V) "
            "— breakdown."
        ))
    if op.v_turn_on > v_max_coss:
        alerts.append(stop(
            f"The datasheet C_oss curve stops at {v_max_coss:.0f} V: P_oss cannot "
            f"be computed at {op.v_turn_on:.0f} V without extrapolating. Extend "
            "`vds_points` / `coss_points` in MOSFET_LIBRARY."
        ))
    for name, value in (("I_rms", op.i_rms), ("Turn-on current", op.i_commutated_on()),
                        ("Turn-off current", op.i_commutated_off())):
        if value > mosfet.i_max:
            alerts.append(warn(
                f"{name} = {value:.0f} A above the current rating "
                f"({mosfet.i_max:.0f} A)."
            ))
    if w_tj.value > mosfet.thermal.t_j_max:
        alerts.append(warn(
            f"Evaluation Tj {w_tj.value:.0f} °C above T_j,max "
            f"({mosfet.thermal.t_j_max:.0f} °C)."
        ))
    # P_rr = Q_rr · V_turn_on · f_sw: under ZVS nothing forces a recovery, so the
    # charge entered costs nothing.
    if w_qrr.value > 0.0 and op.v_turn_on == 0.0:
        alerts.append(warn(
            f"Q_rr = {w_qrr.value:.0f} nC entered but the turn-on voltage is zero "
            "(ZVS): P_rr = Q_rr · V_turn_on · f_sw is zero anyway, the value has "
            "no effect on the budget."
        ))
    return alerts


# ============================================================================ #
#  Report — same blocks as the .py twin, rendered by SRC/OTHERS/terminal.py
# ============================================================================ #


def _report_summary(mosfet, res, th, r_th_label):
    section(None, f"{w_mosfet.value} driven by {w_driver.value}")
    kv("Loss at Tj", f"{res.p_total:.3f} W  (at Tj = {w_tj.value:.0f} °C)")
    kv("R_th junction -> ambient", f"{th.r_th:.1f} °C/W  ({r_th_label})")
    kv("Iterations", f"{th.iterations}  (converged={th.converged})")
    blank()

    t_j_max = mosfet.thermal.t_j_max
    if not th.converged:
        alert("error", "DIVERGES — thermal runaway.", prefix="Tj")
    elif th.t_j_max_exceeded:
        alert("error", f"Tj = {th.t_j:.1f} °C, above T_j,max = {t_j_max:.0f} °C.",
              prefix="Tj")
    else:
        alert("ok", f"Tj = {th.t_j:.1f} °C, {t_j_max - th.t_j:.0f} °C of margin below "
                    f"T_j,max = {t_j_max:.0f} °C.", prefix="Tj")


def _report_thermal_paths(mosfet, active, r_th):
    """One line per branch, then the internals of the PCB planes."""
    section(1, "Thermal paths")
    rows = []
    for d in active:
        r_jc = mosfet.thermal.r_thjc_value(d.placement.value)
        r_ext = d.get_rth()
        rows.append((d.name, f"{r_jc:.2f}", _fmt_r(r_ext), f"{r_jc + r_ext:.1f}"))
    table(["Branch", "R_thJC", "R_ext", "Total [°C/W]"], rows)
    blank()
    kv("Branches in parallel -> R_thJA", f"{r_th:.1f} °C/W")
    kv("(bare datasheet, unused here)", f"{mosfet.thermal.r_thja[0]:.0f} °C/W")

    detail = [
        ("r_spreading_max_cm", "max spreading radius [cm]"),
        ("A_cu_effective_side_cm2", "useful copper, pad side [cm²]"),
        ("A_cu_exposed_side_cm2", "copper exposed to air [cm²]"),
        ("R_conv_side", "R convection, pad side (path A)"),
        ("R_pcb_thru", "R bare substrate under the pad"),
        ("R_vias", "R thermal vias"),
        ("R_through", "R crossing (substrate || vias)"),
        ("R_conv_other", "R convection, far side"),
        ("R_path_B", "path B total"),
        ("R_total", "R_ext of the face (A || B)"),
    ]
    for d in active:
        if not isinstance(d, PCBDissipator):
            continue
        b = d.breakdown()
        blank()
        kv(f"Detail — {d.name}", "[cm, cm², °C/W]")
        table(["Item", "Value"],
              [(label, _fmt_r(b[key])) for key, label in detail if key in b])


def _report_losses(res):
    section(2, "Loss budget [W]")
    dataframe(loss_table(res))
    blank()
    kv("Off-package — driver", f"{res.p_gate_drv * 1e3:.0f} mW")
    kv("Off-package — external gate R", f"{res.p_gate_ext * 1e3:.0f} mW")


def _report_switching(mosfet, res):
    section(3, "Switching")
    table(
        ["Sub-interval", "Duration [ns]", "Slew rate"],
        [("t_ri (current rise)", f"{res.t_ri * 1e9:.1f}",
          f"di/dt on  {res.di_dt_on / 1e9:.2f} A/ns"),
         ("t_fv (voltage fall)", f"{res.t_fv * 1e9:.1f}",
          f"dv/dt on  {res.dv_dt_on / 1e9:.2f} V/ns"),
         ("t_rv (voltage rise)", f"{res.t_rv * 1e9:.1f}",
          f"dv/dt off {res.dv_dt_off / 1e9:.2f} V/ns"),
         ("t_fi (current fall)", f"{res.t_fi * 1e9:.1f}",
          f"di/dt off {res.di_dt_off / 1e9:.2f} A/ns")],
    )
    blank()
    kv("t_on / t_off", f"{res.t_on * 1e9:.1f} ns / {res.t_off * 1e9:.1f} ns")
    kv(f"R_DS(on) at {w_tj.value:.0f} °C",
       f"{res.r_ds_on * 1e3:.3f} mΩ  "
       f"({res.r_ds_on / mosfet.r_ds_on.r_ds_on_25:.2f} x the value at 25 °C)")
    blank()
    # The di/dt used to map the facing diode's datasheet Q_rr onto this design.
    alert("info", f"Q_rr entered: {w_qrr.value:.0f} nC — read it off the facing diode's "
                  f"datasheet at the turn-on di/dt above ({res.di_dt_on / 1e9:.2f} A/ns), "
                  "via BODY_DIODE.q_rr_at_condition().")


def _show(fig, name):
    """Display a figure, and write it to OUTPUT/ when SAVE_PLOTS is on."""
    display(fig)
    if SAVE_PLOTS:
        for path in save_figure(fig, OUTPUT_DIR / name, formats=PLOT_FORMATS,
                                dpi=PLOT_DPI):
            kv("written", str(path))
    plt.close(fig)


def _report_figures(mosfet, res, th):
    section(4, "Figures")
    fig = plot_loss_breakdown(
        res,
        title=f"{w_mosfet.value} — loss breakdown",
        subtitle=f"{w_von.value:.0f} V / {w_ion.value:.0f} A / {w_fsw.value:.0f} kHz "
                 f"at Tj = {w_tj.value:.0f} °C — total {res.p_total:.2f} W",
        theme=PLOT_THEME,
    )
    _show(fig, "single_mosfet_loss_breakdown")

    fig = plot_thermal_iteration(
        th, t_j_max=mosfet.thermal.t_j_max,
        title=f"{w_mosfet.value} — Tj convergence",
        subtitle=f"R_th = {th.r_th:.0f} °C/W, T_amb = {w_tamb.value:.0f} °C",
        theme=PLOT_THEME,
    )
    _show(fig, "single_mosfet_thermal")


# ============================================================================ #
#  Run
# ============================================================================ #


def compute(_=None):
    with out:
        clear_output(wait=True)
        mosfet = load_mosfet(w_mosfet.value)
        driver = load_driver(w_driver.value)
        op = _operating_point()

        r_th, alerts_th, active = _thermal_resistance(mosfet)
        alerts = _checks(mosfet, op) + alerts_th
        for level, message in alerts:
            if level == "stop":
                alert("error", message, prefix="CANNOT COMPUTE")
            else:
                alert("warn", message)
        if any(level == "stop" for level, _ in alerts):
            return

        try:
            res = loss_single_mosfet_at_temp(mosfet, driver, op, t_j=w_tj.value)
        except ValueError as err:
            alert("error", str(err), prefix="CANNOT COMPUTE")
            return

        th = loss_thermal_iteration(mosfet, driver, op, t_ambient=w_tamb.value, r_th=r_th)
        r_th_label = _TH_LABEL[w_th_mode.value] if r_th is not None else "datasheet"

        _report_summary(mosfet, res, th, r_th_label)
        # The branch detail only means something when the branches set R_th.
        if active and r_th is not None:
            _report_thermal_paths(mosfet, active, r_th)
        _report_losses(res)
        _report_switching(mosfet, res)
        _report_figures(mosfet, res, th)


w_go.on_click(compute)
_toggle_rth()
display(panel)
compute()

---
## 4. Reading the result — and what the model does not say

**What is verified.** Energy is conserved: `P_total` is exactly the sum of the six
buckets, the three shares of gate loss add up to $Q_g \Delta V_{gs} f_{sw}$, and
$C_{oss,er}$ does give back $\int_0^V C_{oss}(v)\,v\,dv$.

**Three limits to keep in mind:**

1. **$R_{DS(on)}(T_j)$ is linear, hence optimistic.** At 175 °C the model gives ~1.75 ×
   $R_{25}$ where a $T^{2.3}$ law gives ~2.55 ×, a **46 % gap**. Direct consequence: the
   thermal loop settles lower than reality, and runaway is less likely in the model than
   on the bench. This is the parameter that silently skews everything — re-fit it on the
   datasheet through `alpha_R`.

2. **The $Q_{rr}$ is only worth what you entered.** The model does not invent it, it
   multiplies it by $V_{on} f_{sw}$ — so all the uncertainty sits upstream. If you got it
   by mapping a datasheet test point, remember that at 2 A/ns you are typically 20 ×
   beyond the measurement $di/dt$, and that the exponents $a = b = c = \tfrac12$ are
   engineering values, not physical constants. When recovery weighs heavily in the budget,
   the only real answer is a measurement, or a datasheet $Q_{rr}$ curve read at the right
   place.

3. **$V_{plateau}$ is taken as fixed** although it depends on current ($V_{pl} = V_{th} +
   I_d/g_{fs}$) and on temperature. The switching times inherit that.

**Design points to watch in the results:**

- A high $dv/dt$ at turn-off (> ~5 V/ns) threatens the immunity of the facing device
  (parasitic turn-on through $C_{gd}$) and common mode across the isolation.
- If `P_sw` dominates, play with $R_{gate}$; if it is `P_cond`, it is the part or the
  cooling that needs revisiting.
- The margin under $T_{j,max}$ has to stay comfortable: since the $R_{DS(on)}$ law is
  optimistic, settling 5 °C under the limit is **not** a validated design.